# NLP Pipeline Using NLTK

**Topics covered**

1. Sentence Segmentation  
2. Word Tokenization  
3. Stemming  
4. Lemmatization  
5. Part-of-Speech (POS) Tagging  
6. Named Entity Recognition (NER)  
7. Chunking and Parsed Text  
8. Complete NLP Pipeline  

> **Important:** Stemming and lemmatization are usually treated as two alternative
> normalization methods. In this notebook, they are demonstrated separately.

## 1. Install and Import NLTK

Run the following cell once. Restarting the notebook normally does not require
reinstalling the package.

In [ ]:
%pip install -q nltk

### Download the Required NLTK Resources

Different NLTK releases may use either the older or newer resource names.
The cell below downloads both variants so that the notebook works in common
Jupyter and Google Colab environments.

In [ ]:
import nltk

required_resources = [
    "punkt",
    "punkt_tab",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker",
    "maxent_ne_chunker_tab",
    "words",
]

for resource in required_resources:
    try:
        nltk.download(resource, quiet=True)
        print(f"Ready: {resource}")
    except Exception as error:
        print(f"Could not download {resource}: {error}")

In [ ]:
from collections import Counter
import string

from nltk import ne_chunk, pos_tag
from nltk.chunk import tree2conlltags
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tree import Tree

## Sample Text

In [ ]:
sample_text = (
    "Dr. Asha works at Toyota Connected India in Bengaluru. "
    "She is developing an NLP application for students. "
    "The team will present the project on Monday."
)

print(sample_text)

# Task 1 — Sentence Segmentation

**Sentence segmentation** divides a paragraph into individual sentences.

In [ ]:
sentences = sent_tokenize(sample_text)

for number, sentence in enumerate(sentences, start=1):
    print(f"Sentence {number}: {sentence}")

# Task 2 — Word Tokenization

**Tokenization** divides text into smaller units called tokens. A token may be
a word, number, or punctuation mark.

In [ ]:
tokens = word_tokenize(sample_text)

print("All tokens:")
print(tokens)

alphabetic_tokens = [token.lower() for token in tokens if token.isalpha()]

print("\nAlphabetic tokens:")
print(alphabetic_tokens)

# Task 3 — Stemming

**Stemming** removes prefixes or suffixes using rule-based processing. The
resulting stem may not be a valid dictionary word.

Example: `studies → studi`

In [ ]:
porter = PorterStemmer()

words_for_stemming = [
    "connect",
    "connected",
    "connecting",
    "connection",
    "studies",
    "studying",
    "studied",
]

for word in words_for_stemming:
    print(f"{word:12} -> {porter.stem(word)}")

# Task 4 — Lemmatization

**Lemmatization** finds the dictionary base form of a word. Supplying the
correct grammatical category improves the result.

Examples:

- `cars → car` as a noun
- `running → run` as a verb
- `better → good` as an adjective

In [ ]:
lemmatizer = WordNetLemmatizer()

lemma_examples = [
    ("cars", "n"),
    ("children", "n"),
    ("running", "v"),
    ("studied", "v"),
    ("better", "a"),
]

for word, wordnet_pos in lemma_examples:
    lemma = lemmatizer.lemmatize(word, pos=wordnet_pos)
    print(f"{word:10} ({wordnet_pos}) -> {lemma}")

# Task 5 — Part-of-Speech Tagging

**POS tagging** assigns a grammatical category to every token.

Common Penn Treebank tags:

| Tag | Meaning |
|---|---|
| NN | Singular noun |
| NNS | Plural noun |
| NNP | Proper noun |
| VB | Base-form verb |
| VBD | Past-tense verb |
| VBG | Verb ending in `-ing` |
| JJ | Adjective |
| RB | Adverb |

In [ ]:
pos_sentence = "The intelligent students are building useful language applications."
pos_tokens = word_tokenize(pos_sentence)
tagged_tokens = pos_tag(pos_tokens)

for token, tag in tagged_tokens:
    print(f"{token:15} {tag}")

# Task 6 — Named Entity Recognition

**Named Entity Recognition (NER)** identifies important real-world entities,
such as:

- `PERSON`
- `ORGANIZATION`
- `GPE` — geopolitical entity, such as a country or city
- `LOCATION`
- `FACILITY`

NLTK's `ne_chunk` produces a tree containing the detected entities.

In [ ]:
ner_text = (
    "Sundar Pichai works at Google. "
    "He visited India and met researchers in Chennai."
)

ner_tokens = word_tokenize(ner_text)
ner_pos_tags = pos_tag(ner_tokens)
ner_tree = ne_chunk(ner_pos_tags)

print(ner_tree)

In [ ]:
def extract_named_entities(chunk_tree):
    entities = []

    for node in chunk_tree:
        if isinstance(node, Tree):
            entity_text = " ".join(word for word, tag in node.leaves())
            entities.append((entity_text, node.label()))

    return entities


entities = extract_named_entities(ner_tree)

print("Named entities:")
for entity, entity_type in entities:
    print(f"{entity:25} -> {entity_type}")

# Task 7 — Chunking and Parsed Text

**Chunking** groups POS-tagged words into meaningful phrases. The following
simple grammar identifies noun phrases (`NP`).

Grammar:

```text
NP: {<DT>?<JJ.*>*<NN.*>+}
```

It means:

- optional determiner (`DT`);
- zero or more adjectives (`JJ...`);
- one or more nouns (`NN...`).

In [ ]:
chunk_text = "The skilled engineer designed an intelligent monitoring system."
chunk_tokens = word_tokenize(chunk_text)
chunk_tags = pos_tag(chunk_tokens)

noun_phrase_grammar = r"NP: {<DT>?<JJ.*>*<NN.*>+}"
chunk_parser = nltk.RegexpParser(noun_phrase_grammar)
parsed_tree = chunk_parser.parse(chunk_tags)

print(parsed_tree)

In [ ]:
noun_phrases = [
    " ".join(word for word, tag in subtree.leaves())
    for subtree in parsed_tree.subtrees(lambda tree: tree.label() == "NP")
]

print("Noun phrases:", noun_phrases)

## Exercise

Write an NLTK program that generates an NLP report containing:

- Number of sentences
- Number of alphabetic tokens
- Five most frequent lowercase tokens
- Stems of all alphabetic tokens
- Named entities

In [ ]:
import nltk
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from nltk import pos_tag, ne_chunk
from nltk.tree import Tree

text = """
Sundar Pichai works at Google in California.
He visited Chennai for an artificial intelligence conference.
Google organized the conference for students in Chennai.
"""

# 1. Split the text into sentences
# Write your code here


# 2. Tokenize the text into words
# Write your code here


# 3. Keep only alphabetic tokens
# Write your code here


# 4. Convert the alphabetic tokens to lowercase
# Write your code here


# 5. Find the five most frequent tokens
# Write your code here


# 6. Find the stems of all alphabetic tokens
# Write your code here


# 7. Find the named entities
# Write your code here


# 8. Print the NLP report
# Write your code here